In [1]:
%pip install einops
import torch
import torch.nn as nn
from einops import rearrange, repeat
import torch.optim as optim

print(f"PyTorch version: {torch.__version__}")


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
PyTorch version: 2.12.1+cu130


In [2]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class Attention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=64, dropout=0.):
        super().__init__()
        inner_dim = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        h = self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)

        dots = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        attn = dots.softmax(dim=-1)
        
        out = torch.matmul(attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        return self.to_out(out)

In [3]:
class DividedSpaceTimeBlock(nn.Module):
    def __init__(self, dim, heads, dim_head, mlp_dim, dropout=0.):
        super().__init__()
        # 1. Khối Thời gian (Temporal)
        self.time_norm1 = nn.LayerNorm(dim)
        self.time_attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        
        # 2. Khối Không gian (Spatial)
        self.space_norm1 = nn.LayerNorm(dim)
        self.space_attn = Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout)
        
        # 3. Khối MLP tinh chỉnh
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_dim, drop=dropout)

    def forward(self, x, f):
        b, t, n, d = x.shape  # b: batch, t: thời gian, n: không gian, d: số chiều
        
        # --- BƯỚC 1: Chú ý Thời gian ---
        # Gộp Batch và Không gian để mạng CHỈ nhìn thấy trục Thời gian
        x_time = rearrange(x, 'b t n d -> (b n) t d')
        x_time = x_time + self.time_attn(self.time_norm1(x_time))
        x = rearrange(x_time, '(b n) t d -> b t n d', b=b, n=n)

        # --- BƯỚC 2: Chú ý Không gian ---
        # Gộp Batch và Thời gian để mạng CHỈ nhìn thấy trục Không gian
        x_space = rearrange(x, 'b t n d -> (b t) n d')
        x_space = x_space + self.space_attn(self.space_norm1(x_space))
        x = rearrange(x_space, '(b t) n d -> b t n d', b=b, t=t)

        # --- BƯỚC 3: Mạng FFN (MLP) ---
        x_mlp = rearrange(x, 'b t n d -> (b t n) d')
        x_mlp = x_mlp + self.mlp(self.norm2(x_mlp))
        
        return rearrange(x_mlp, '(b t n) d -> b t n d', b=b, t=t, n=n)

In [4]:
class TimeSformer(nn.Module):
    def __init__(self, image_size=224, patch_size=16, num_frames=8, num_classes=2, 
                 dim=256, depth=6, heads=8, dim_head=64, mlp_dim=512, dropout=0.1):
        super().__init__()
        num_patches = (image_size // patch_size) ** 2
        patch_dim = 3 * patch_size ** 2
        self.num_frames = num_frames
        
        # Nhúng mảnh ảnh thành Vector
        self.to_patch_embedding = nn.Linear(patch_dim, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, 1, dim))
        
        # Tọa độ Không gian và Thời gian
        self.pos_embedding = nn.Parameter(torch.randn(1, 1, num_patches + 1, dim))
        self.time_embedding = nn.Parameter(torch.randn(1, num_frames, 1, dim))

        # Các lớp Attention
        self.layers = nn.ModuleList([
            DividedSpaceTimeBlock(dim, heads, dim_head, mlp_dim, dropout)
            for _ in range(depth)
        ])

        # Phân loại đầu ra
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, video):
        b, c, f, h, w = video.shape
        p = 16 
        
        # Băm video
        x = rearrange(video, 'b c f (h p1) (w p2) -> b f (h w) (c p1 p2)', p1=p, p2=p)
        x = self.to_patch_embedding(x)
        b, t, n, d = x.shape
        
        cls_tokens = repeat(self.cls_token, '() () () d -> b t () d', b=b, t=t)
        x = torch.cat((cls_tokens, x), dim=2)
        
        # Cộng tọa độ
        x = x + self.pos_embedding[:, :, :n+1]
        x[:, :, 1:] = x[:, :, 1:] + self.time_embedding[:, :t, :]

        # Xử lý qua mạng
        for layer in self.layers:
            x = layer(x, t)

        # Lấy token CLS và trung bình hóa theo trục thời gian
        cls_out = x[:, :, 0, :].mean(dim=1)
        return self.mlp_head(cls_out)

In [5]:
def apply_spatial_freeze_strategy(model):
    print("\n--- KÍCH HOẠT CHIẾN LƯỢC TỐI ƯU HÓA ---")
    frozen_params = 0
    trainable_params = 0

    for name, param in model.named_parameters():
        # Chỉ MỞ KHÓA các lớp học Thời gian và đầu ra
        if 'time_attn' in name or 'time_norm' in name or 'time_embedding' in name or 'mlp_head' in name:
            param.requires_grad = True
            trainable_params += param.numel()
        # ĐÓNG BĂNG toàn bộ phần Không gian
        else:
            param.requires_grad = False
            frozen_params += param.numel()

    total = frozen_params + trainable_params
    print(f"Tổng tham số: {total / 1e6:.2f} M")
    print(f"  [KHÓA] Tham số bị đóng băng: {frozen_params / 1e6:.2f} M ({frozen_params/total*100:.1f}%)")
    print(f"  [MỞ] Tham số được huấn luyện: {trainable_params / 1e6:.2f} M ({trainable_params/total*100:.1f}%)")
    print("----------------------------------------\n")

In [6]:
# 1. Khởi tạo mô hình cực nhẹ cho Edge AI
model = TimeSformer(
    image_size=224, patch_size=16, num_frames=8, num_classes=2, 
    dim=256, depth=6, heads=8, mlp_dim=512
)

# 2. GIAI ĐOẠN 1: Tưởng tượng bạn đã Train trên Dataset Indoor Fire
print(">>> GIAI ĐOẠN 1: Đã hoàn tất huấn luyện mạng ViT trên ảnh tĩnh 'Indoor Fire Smoke'.")
print(">>> Giả lập nạp file trọng số 'vit_fire_smoke.pth' thành công.\n")

# 3. GIAI ĐOẠN 2: Fine-tuning trên Video
# Khóa chặt kiến thức không gian
apply_spatial_freeze_strategy(model)

# Khai báo Optimizer: CHỈ đưa vào những tham số mở khóa (requires_grad=True)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

# Tạo batch video giả lập: Batch=2, Channels=3, Frames=8, Height=224, Width=224
# Đây chính là mảng tensor bạn tự tay gắp bằng thuật toán lấy mẫu nhảy cóc!
dummy_video = torch.randn(2, 3, 8, 224, 224)

# Nhãn giả lập: Video 1 có cháy (Class 1), Video 2 không cháy (Class 0)
targets = torch.tensor([1, 0])

print(">>> BẮT ĐẦU VÒNG LẶP HUẤN LUYỆN (Chỉ học Thời gian)\n")
epochs = 5
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    # Lan truyền xuôi (Forward)
    outputs = model(dummy_video)
    
    # Tính Loss
    loss = loss_fn(outputs, targets)
    
    # Lan truyền ngược (Backward) - Nó sẽ rất nhanh vì 70% mạng không cần đạo hàm!
    loss.backward()
    
    # Cập nhật trọng số
    optimizer.step()
    
    # Log quá trình
    predicted_classes = outputs.argmax(dim=-1).tolist()
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f} | AI dự đoán: {predicted_classes} | Nhãn thật: {targets.tolist()}")

print("\n>>> KẾT THÚC! Mô hình đã sẵn sàng để xuất ra ONNX định dạng nhẹ.")

>>> GIAI ĐOẠN 1: Đã hoàn tất huấn luyện mạng ViT trên ảnh tĩnh 'Indoor Fire Smoke'.
>>> Giả lập nạp file trọng số 'vit_fire_smoke.pth' thành công.


--- KÍCH HOẠT CHIẾN LƯỢC TỐI ƯU HÓA ---
Tổng tham số: 8.13 M
  [KHÓA] Tham số bị đóng băng: 4.98 M (61.2%)
  [MỞ] Tham số được huấn luyện: 3.15 M (38.8%)
----------------------------------------

>>> BẮT ĐẦU VÒNG LẶP HUẤN LUYỆN (Chỉ học Thời gian)

Epoch [1/5] | Loss: 0.7372 | AI dự đoán: [0, 0] | Nhãn thật: [1, 0]
Epoch [2/5] | Loss: 0.7946 | AI dự đoán: [1, 1] | Nhãn thật: [1, 0]
Epoch [3/5] | Loss: 0.7089 | AI dự đoán: [1, 1] | Nhãn thật: [1, 0]
Epoch [4/5] | Loss: 0.6978 | AI dự đoán: [0, 0] | Nhãn thật: [1, 0]
Epoch [5/5] | Loss: 0.7499 | AI dự đoán: [0, 0] | Nhãn thật: [1, 0]

>>> KẾT THÚC! Mô hình đã sẵn sàng để xuất ra ONNX định dạng nhẹ.


In [7]:
%pip install opencv-python
import os
import cv2
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

def custom_yolo_to_classification_generator(image_dir, batch_size=32, shuffle=True):
    """
    Hàm sinh dữ liệu (Generator) code chay, không dùng DataLoader.
    Chuyển đổi nhãn YOLO bounding box thành nhãn Multi-label Classification.
    Target output: [has_fire, has_smoke] (ví dụ: [1.0, 0.0], [1.0, 1.0])
    """
    # Đường dẫn thư mục nhãn tương ứng (thay 'images' thành 'labels')
    label_dir = image_dir.replace('images', 'labels')
    
    # Lấy danh sách toàn bộ ảnh
    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    if shuffle:
        random.shuffle(image_files)

    batch_imgs = []
    batch_targets = []

    for img_file in image_files:
        # 1. XỬ LÝ ẢNH
        img_path = os.path.join(image_dir, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        
        # BGR -> RGB, Resize về 224x224, Chuẩn hóa 0-1
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))
        img = img.astype(np.float32) / 255.0
        
        # Đổi trục từ (H, W, C) sang (C, H, W) cho PyTorch
        img = np.transpose(img, (2, 0, 1))

        # 2. XỬ LÝ NHÃN (Chuyển Bounding Box thành Classification)
        target = [0.0, 0.0] # [fire, smoke]
        txt_file = os.path.splitext(img_file)[0] + '.txt'
        txt_path = os.path.join(label_dir, txt_file)

        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) > 0:
                        class_id = int(parts[0])
                        if class_id == 0:
                            target[0] = 1.0 # Có lửa
                        elif class_id == 1:
                            target[1] = 1.0 # Có khói
        
        batch_imgs.append(img)
        batch_targets.append(target)

        # 3. YIELD BATCH KHI ĐỦ SỐ LƯỢNG
        if len(batch_imgs) == batch_size:
            # Gộp thành tensor: shape (Batch, Channels, Height, Width)
            tensor_imgs = torch.tensor(np.array(batch_imgs), dtype=torch.float32)
            tensor_targets = torch.tensor(np.array(batch_targets), dtype=torch.float32)
            
            # Reset mảng chứa
            batch_imgs = []
            batch_targets = []
            
            yield tensor_imgs, tensor_targets

    # Yield phần dư cuối cùng (nếu có)
    if len(batch_imgs) > 0:
        tensor_imgs = torch.tensor(np.array(batch_imgs), dtype=torch.float32)
        tensor_targets = torch.tensor(np.array(batch_targets), dtype=torch.float32)
        yield tensor_imgs, tensor_targets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 181.3 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


ImportError: libGL.so.1: cannot open shared object file: No such file or directory

In [ ]:
# Kích hoạt thiết bị (GPU trên Modal)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên thiết bị: {device}")

# Khởi tạo mô hình ở "Chế độ Ảnh tĩnh" (num_frames=1)
# Kích thước siêu nhẹ: dim=256, depth=6
model = TimeSformer(
    image_size=224, 
    patch_size=16, 
    num_frames=1,    # QUAN TRỌNG: 1 frame biến nó thành mạng ViT
    num_classes=2,   # 2 class: Lửa, Khói
    dim=256, 
    depth=6, 
    heads=8, 
    mlp_dim=512
).to(device)

# Sử dụng BCEWithLogitsLoss cho bài toán Multi-label (vì 1 ảnh có thể vừa có khói vừa có lửa)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

print("Đã khởi tạo mô hình Spatial Attention thành công.")

Đang chạy trên thiết bị: cuda
Đã khởi tạo mô hình Spatial Attention thành công.


In [ ]:
import os

data_path = '/data/merged_dataset'

if os.path.exists(data_path):
    print("✅ Đã kết nối Volume thành công!")
    print("Danh sách thư mục bên trong:")
    print(os.listdir(data_path))
    
    # Check thử thư mục train
    train_images = os.path.join(data_path, 'train/images')
    if os.path.exists(train_images):
        num_imgs = len(os.listdir(train_images))
        print(f"👉 Tìm thấy {num_imgs} ảnh trong tập Train.")
else:
    print("❌ Vẫn chưa thấy dữ liệu. Hãy kiểm tra lại đường dẫn mount Volume!")

❌ Vẫn chưa thấy dữ liệu. Hãy kiểm tra lại đường dẫn mount Volume!


In [ ]:
# Đường dẫn trên Modal
train_dir = '/data/merged_dataset/train/images'
val_dir = '/data/merged_dataset/val/images'

epochs = 10
batch_size = 64

print(f"{'='*50}")
print(" BẮT ĐẦU GIAI ĐOẠN 1: SPATIAL PRE-TRAINING")
print(f"{'='*50}")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    batch_count = 0
    
    # Khởi tạo generator thủ công
    train_gen = custom_yolo_to_classification_generator(train_dir, batch_size=batch_size, shuffle=True)
    
    for batch_imgs, batch_targets in train_gen:
        # TimeSformer yêu cầu đầu vào là Video 5D: (Batch, Channels, Frames, Height, Width)
        # Hiện tại batch_imgs đang là 4D: (Batch, Channels, Height, Width)
        # Ta dùng unsqueeze(2) để thêm chiều Frames (F=1) vào giữa
        batch_imgs = batch_imgs.unsqueeze(2).to(device)
        batch_targets = batch_targets.to(device)
        
        optimizer.zero_grad()
        
        # Lan truyền xuôi
        outputs = model(batch_imgs)
        
        # Tính loss và Lan truyền ngược
        loss = loss_fn(outputs, batch_targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        batch_count += 1
        
        # In log mỗi 50 batch
        if batch_count % 50 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Batch {batch_count} | Loss: {loss.item():.4f}")
            
    epoch_loss = running_loss / batch_count
    print(f"---> Hết Epoch {epoch+1} | Loss trung bình: {epoch_loss:.4f}\n")

# Lưu lại trọng số kiến thức Không gian
save_path = 'spatial_fire_smoke_weights.pth'
torch.save(model.state_dict(), save_path)
print(f"Đã lưu trọng số khối Spatial tại: {save_path}")
print("Sẵn sàng cho Giai đoạn 2 (Temporal Fine-tuning với Video)!")

 BẮT ĐẦU GIAI ĐOẠN 1: SPATIAL PRE-TRAINING


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/data/merged_dataset/train/images'